In [63]:
# https://dblp.org/faq/How+to+use+the+dblp+search+API.html
# 
# https://dblp.org/search/publ/api for publication queries
# https://dblp.org/search/author/api for author queries
# https://dblp.org/search/venue/api for venue queries

In [ ]:
# Format of the query. Add $ for exact matches
# 
#  https://dblp.org/search/author/api?q=David%20Silver&format=json

In [43]:
import requests, re, unicodedata
import json
from pprint import pprint

In [ ]:
def ensure_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

try:
    from rapidfuzz.fuzz import ratio as fuzz_ratio
except ImportError:
    fuzz_ratio = None

def normalize_title(s):
    s = unicodedata.normalize('NFKD', s)
    s = re.sub(r'[^A-Za-z0-9 ]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip().lower()

def tokens(s):
    return set(normalize_title(s).split())

def jaccard(a, b):
    if not a or not b: return 0.0
    ia = a & b
    ua = a | b
    return len(ia) / len(ua) if ua else 0.0

def looks_main_track(key):
    if not key: 
        return False
    # Adjust patterns for the domains; example prefers main conference tracks
    return bool(re.search(r'^conf/(nips|neurips|acl|iclr|icml|ijcai)/', key))

def score_candidate(c, mention_norm, mention_toks, year_hint=None, venue_hints=None, author_hints=None):
    title = c.get("title") or ""
    key = c.get("key") or ""
    year = c.get("year")
    doi = c.get("doi")
    authors = [a.get("name","") for a in c.get("authors", [])]
    venue = c.get("venue","") or ""

    title_norm = normalize_title(title)
    title_toks = tokens(title)

    # Exact normalized title assign high score
    if title and title_norm == mention_norm:
        base = 1000000
    else:
        # Fuzzy title similarity
        jac = jaccard(mention_toks, title_toks)
        edit = (fuzz_ratio(mention_norm, title_norm)/100.0) if fuzz_ratio else jac
        base = 800*edit + 900*jac

    # Side signals
    bonus = 0
    if year_hint and year and str(year_hint) == str(year):
        bonus += 80
    if venue_hints:
        vt = tokens(venue)
        tt = tokens(title)

        vh_toks = set()
        for vh in venue_hints:
            vh_toks |= tokens(vh)

        overlap = len(vh_toks & vt) + 0.5 * len(vh_toks & tt)
        bonus += int(25 * min(overlap, 2))
    
    if author_hints:
        a_set = {a.lower() for a in authors if a}
        overlap = sum(1 for ah in author_hints if any(part in a for a in a_set for part in ah.lower().split()))
        bonus += 40 * min(overlap, 2)
    
    if doi:  # stable id present
        bonus += 10
    
    if looks_main_track(key):
        bonus += 15

    return base + bonus

def rank_candidates(candidates, mention, year_hint=None, venue_hints=None, author_hints=None):
    mention_norm = normalize_title(mention)
    mention_toks = tokens(mention)
    scored = []
    for c in candidates:
        s = score_candidate(c, mention_norm, mention_toks, year_hint, venue_hints, author_hints)
        scored.append((s, c))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [c for s, c in scored]

In [45]:
def parse_dblp_hits(json_obj):
    hits = (((json_obj or {}).get("result") or {}).get("hits") or {}).get("hit") or []
    if isinstance(hits, dict):  # DBLP may return a single object instead of a list
        hits = [hits]
    parsed = []
    for h in hits:
        info = (h or {}).get("info") or {}
        key = info.get("key")
        title = info.get("title")
        year = info.get("year")
        
        venue = info.get("venue") or info.get("journal") or info.get("booktitle")

        ee = info.get("ee")
        ee_list = ensure_list(ee)

        authors_raw = (info.get("authors") or {}).get("author") # authors can be a dict with list/dict under 'author', or a single author
        authors = []
        for a in ensure_list(authors_raw):
            if isinstance(a, dict):
                name = a.get("text") or a.get("name") or a.get("label")
                pid = a.get("@pid") or a.get("pid")
            else:
                name, pid = str(a), None
            authors.append({"name": name, "pid": pid})

        doi = info.get("doi")
        rec_url = f"https://dblp.org/rec/{key}" if key else None  # stable IRI for the record

        parsed.append({
            "key": key,
            "rec_url": rec_url,
            "title": title,
            "year": year,
            "venue": venue,
            "ee": ee_list,
            "doi": doi,
            "authors": authors,
        })
    return parsed

In [ ]:
#base_url = "https://dblp.org/search/author/api"
base_url = "https://dblp.org/search/publ/api"
#search_query = "David Silver$" # Remove $ for non-exact matches as well
#search_query = "Attention$ is$ all$ you$ need$ year:2017"

search_query = "Attention is all you need" # phrase search is disabled, so a longer title that contains all those exact words can rank above the Vaswani paper unless we add more constraints or post-filter the results
#search_query = "Deep Learning"

In [61]:
params = {
    'q': search_query,
    'h': 10, # max cap is 1000
    'format': 'json'
}

In [62]:
resp = requests.get(base_url, params=params, timeout=20)
resp.raise_for_status()
data = resp.json()

In [63]:
candidates = parse_dblp_hits(data)
print(len(candidates))
for candidate in candidates:
    print(candidate)

10
{'key': 'conf/icse-deeptest/2025', 'rec_url': 'https://dblp.org/rec/conf/icse-deeptest/2025', 'title': 'IEEE/ACM International Workshop on Deep Learning for Testing and Testing for Deep Learning, DeepTest@ICSE 2025, Ottawa, ON, Canada, May 3, 2025', 'year': '2025', 'venue': 'DeepTest', 'ee': ['https://doi.org/10.1109/DeepTest66595.2025'], 'doi': '10.1109/DEEPTEST66595.2025', 'authors': []}
{'key': 'phd/hal/Simonian24', 'rec_url': 'https://dblp.org/rec/phd/hal/Simonian24', 'title': 'Study of IA methods in data analytics (machine learning / deep learning) to improve the content of computer science MOOCs and contribution to the design of the FUN (France Université Numérique) data processing chain. (Étude des méthodes IA d&apos;analyse de données (machine learning / deep learning) pour l&apos;amélioration du contenu des MOOCs informatiques et contribution à la conception de la chaîne de traitement des données de FUN (France Université Numérique)).', 'year': '2024', 'venue': None, 'ee': 

In [ ]:
ranked = rank_candidates(candidates, mention="Deep Learning")#, year_hint=2017, venue_hints=["NeurIPS","NIPS"])
top = ranked[0]
#print(top)

rank = 1
for entity in ranked:
    print(rank, entity)
    rank += 1

1 {'key': 'conf/nips/VaswaniSPUJGKP17', 'rec_url': 'https://dblp.org/rec/conf/nips/VaswaniSPUJGKP17', 'title': 'Attention is All you Need.', 'year': '2017', 'venue': 'NIPS', 'ee': ['https://proceedings.neurips.cc/paper/2017/hash/3f5ee243547dee91fbd053c1c4a845aa-Abstract.html'], 'doi': None, 'authors': [{'name': 'Ashish Vaswani', 'pid': '26/9012'}, {'name': 'Noam Shazeer', 'pid': '80/4668'}, {'name': 'Niki Parmar', 'pid': '202/2051'}, {'name': 'Jakob Uszkoreit', 'pid': '87/4805'}, {'name': 'Llion Jones', 'pid': '184/3736'}, {'name': 'Aidan N. Gomez', 'pid': '202/2262'}, {'name': 'Lukasz Kaiser', 'pid': '39/1762'}, {'name': 'Illia Polosukhin', 'pid': '184/3747'}]}
2 {'key': 'journals/corr/VaswaniSPUJGKP17', 'rec_url': 'https://dblp.org/rec/journals/corr/VaswaniSPUJGKP17', 'title': 'Attention Is All You Need.', 'year': '2017', 'venue': 'CoRR', 'ee': ['http://arxiv.org/abs/1706.03762'], 'doi': None, 'authors': [{'name': 'Ashish Vaswani', 'pid': '26/9012'}, {'name': 'Noam Shazeer', 'pid': '

In [ ]:
# # Prefer exact normalized title match; otherwise fall back to top hit
# mention = "Attention Is All You Need"
# mention_norm = normalize_title(mention)
# exact = [p for p in candidates if p["title"] and normalize_title(p["title"]) == mention_norm]
# chosen = exact[0] if exact else (candidates[0] if candidates else None)

# print("Chosen record:", chosen)
# print("Authors:", [a["name"] for a in (chosen["authors"] if chosen else []) if a["name"]])
# print("DBLP IRI:", chosen["rec_url"] if chosen else None)

Chosen record: {'key': 'journals/corr/abs-2407-15516', 'rec_url': 'https://dblp.org/rec/journals/corr/abs-2407-15516', 'title': 'Attention Is All You Need But You Don&apos;t Need All Of It For Inference of Large Language Models.', 'year': '2024', 'venue': 'CoRR', 'ee': ['https://doi.org/10.48550/arXiv.2407.15516'], 'doi': '10.48550/ARXIV.2407.15516', 'authors': [{'name': 'Georgy Tyukin', 'pid': '375/1509'}, {'name': 'Gbètondji J.-S. Dovonon', 'pid': '342/2924'}, {'name': 'Jean Kaddour', 'pid': '232/9592'}, {'name': 'Pasquale Minervini', 'pid': '58/10142'}]}
Authors: ['Georgy Tyukin', 'Gbètondji J.-S. Dovonon', 'Jean Kaddour', 'Pasquale Minervini']
DBLP IRI: https://dblp.org/rec/journals/corr/abs-2407-15516


In [ ]:
# print(f"Querying DBLP for: '{search_query}'\n")

Querying DBLP for: 'Attention$ is$ all$ you$ need$'



In [ ]:
# try:
#     response = requests.get(base_url, params=params)

#     response.raise_for_status() # Check if the request was successful. This will raise an exception for bad responses (like 404 or 500)
    

#     data = response.json()

#     #print("--- Full JSON Response ---")
#     #pprint(data)
    
# except requests.exceptions.HTTPError as http_err:
#     print(f"HTTP error occurred: {http_err}")
# except requests.exceptions.RequestException as err:
#     print(f"An error occurred: {err}")

In [ ]:
# # ... (inside the try block after 'data = response.json()')

# print("\n--- Extracted Candidates ---")

# # Navigate to the list of 'hits'
# # It's good to use .get() to avoid errors if a key doesn't exist
# hits = data.get('result', {}).get('hits', {}).get('hit', [])

# if not hits:
#     print("No authors found for this query.")
# else:
#     # Loop over each 'hit' (each potential author)
#     for hit in hits:
#         info = hit.get('info', {})
        
#         print(info)

#         # Extract the key data points
#         author_name = info.get('author')
#         dblp_url = info.get('url') # This contains the persistent ID (PID)
        
#         # The affiliation/note is often in 'notes'
#         affiliation = None
#         if info.get('notes'):
#             # 'notes' can be a list or a single dict
#             notes = info['notes']
#             if isinstance(notes, list):
#                 affiliation = notes[0].get('text')
#             elif isinstance(notes, dict):
#                 affiliation = notes.get('text')
        
#         print(f"Candidate: {author_name}")
#         print(f"  URL (ID): {dblp_url}")
#         print(f"  Context: {affiliation}")
#         print("-" * 10)


--- Extracted Candidates ---
{'authors': {'author': [{'@pid': '375/1509', 'text': 'Georgy Tyukin'}, {'@pid': '342/2924', 'text': 'Gbètondji J.-S. Dovonon'}, {'@pid': '232/9592', 'text': 'Jean Kaddour'}, {'@pid': '58/10142', 'text': 'Pasquale Minervini'}]}, 'title': 'Attention Is All You Need But You Don&apos;t Need All Of It For Inference of Large Language Models.', 'venue': 'CoRR', 'volume': 'abs/2407.15516', 'year': '2024', 'type': 'Informal and Other Publications', 'access': 'open', 'key': 'journals/corr/abs-2407-15516', 'doi': '10.48550/ARXIV.2407.15516', 'ee': 'https://doi.org/10.48550/arXiv.2407.15516', 'url': 'https://dblp.org/rec/journals/corr/abs-2407-15516'}
Candidate: None
  URL (ID): https://dblp.org/rec/journals/corr/abs-2407-15516
  Context: None
----------
{'authors': {'author': {'@pid': '397/7878', 'text': 'M. Murat Yaslioglu'}}, 'title': 'Attention is All You Need Until You Need Retention.', 'venue': 'CoRR', 'volume': 'abs/2501.09166', 'year': '2025', 'type': 'Informa